# Part 11: XBRL validation

Checks every number in the extracted statement tables (traditional path from Part 2, Docling from Part 4) against the filing's own inline XBRL facts, loaded with Arelle by `src/xbrl.py`.

Run `dvc repro xbrl` (or `python src/xbrl.py`) first; this notebook only reads its outputs in `data/xbrl/`.

In [1]:
import json
import pandas as pd

summary = json.load(open('../data/xbrl/summary.json'))
comp = pd.read_csv('../data/xbrl/comparison.csv')
facts = pd.read_csv('../data/xbrl/facts.csv')
print(f'{len(facts)} numeric facts, {len(comp)} compared numbers')

1750 numeric facts, 908 compared numbers


## Match rate per path and statement

In [2]:
rows = [{'path': p, 'statement': s, **{k: v for k, v in m.items() if k != 'status_counts'}}
        for p, stmts in summary.items() for s, m in stmts.items()]
pd.DataFrame(rows).set_index(['path', 'statement'])

numbers  mapped  match  match_rate  \
path        statement                                                  
docling     ALL                       452     452    450      0.9956   
            balance                   108     108    108      1.0000   
            cash_flows                170     170    168      0.9882   
            comprehensive_income       60      60     60      1.0000   
            income                    114     114    114      1.0000   
traditional ALL                       456     456    456      1.0000   
            balance                   108     108    108      1.0000   
            cash_flows                174     174    174      1.0000   
            comprehensive_income       60      60     60      1.0000   
            income                    114     114    114      1.0000   

                                  match_rate_of_mapped  
path        statement                                   
docling     ALL                                 0.9956  
            balance                             1.0000  
            cash_flows                          0.9882  
            comprehensive_income                1.0000  
            income                              1.0000  
traditional ALL                                 1.0000  
            balance                             1.0000  
            cash_flows                          1.0000  
            comprehensive_income                1.0000  
            income                              1.0000

## Status counts

`match_negated`: the line is printed with a negated label (for example a cash outflow in parentheses) while XBRL stores the positive amount.

In [3]:
comp.groupby(['path', 'statement', 'status']).size().unstack(fill_value=0)

status                            match  match_negated  pdf_missing
path        statement                                              
docling     balance                 108              0            0
            cash_flows               96             72            2
            comprehensive_income     48             12            0
            income                  114              0            0
traditional balance                 108              0            0
            cash_flows              102             72            0
            comprehensive_income     48             12            0
            income                  114              0            0

## How each label was mapped to a concept

curated = `config/label_map.yaml`; linkbase = the filing's own labels; fuzzy = close spelling match.

In [4]:
comp['method'] = comp['map_method'].str.split(':').str[0]
comp.groupby(['path', 'method']).size().unstack(fill_value=0)

method,curated,found by traditional only,linkbase
path,,,
docling,146,2,304
traditional,146,0,310


## Anything that did not match

In [5]:
bad = comp[~comp['status'].isin(['match', 'match_negated'])]
print(f'{len(bad)} non-matching numbers')
bad[['stem', 'path', 'statement', 'label', 'year', 'value', 'concept', 'xbrl_value', 'status']]

2 non-matching numbers


,stem,path,statement,label,year,value,concept,xbrl_value,status
906,AAPL_10K_20240928,docling,cash_flows,"Cash, cash equivalents, and restricted cash an...",2022,NaN,us-gaap:CashCashEquivalentsRestrictedCashAndRe...,3.592900e+10,pdf_missing
907,AAPL_10K_20250927,docling,cash_flows,"Cash, cash equivalents, and restricted cash an...",2023,NaN,us-gaap:CashCashEquivalentsRestrictedCashAndRe...,2.497700e+10,pdf_missing


## Numbers found by one path but not the other

In [6]:
k = ['stem', 'statement', 'label', 'occurrence', 'year']
t = comp[comp.path == 'traditional'][k]
d = comp[comp.path == 'docling'][k]
m = t.merge(d, on=k, how='outer', indicator=True)
m[m['_merge'] != 'both']

,stem,statement,label,occurrence,year,_merge
44,AAPL_10K_20240928,balance,Total liabilities and shareholders' equity,1,2023,right_only
45,AAPL_10K_20240928,balance,Total liabilities and shareholders' equity,1,2024,right_only
46,AAPL_10K_20240928,balance,Total liabilities and shareholders’ equity,1,2023,left_only
47,AAPL_10K_20240928,balance,Total liabilities and shareholders’ equity,1,2024,left_only
52,AAPL_10K_20240928,balance,Total shareholders' equity,1,2023,right_only
53,AAPL_10K_20240928,balance,Total shareholders' equity,1,2024,right_only
54,AAPL_10K_20240928,balance,Total shareholders’ equity,1,2023,left_only
55,AAPL_10K_20240928,balance,Total shareholders’ equity,1,2024,left_only
77,AAPL_10K_20240928,cash_flows,"Cash, cash equivalents, and restricted cash an...",1,2023,left_only
78,AAPL_10K_20240928,cash_flows,"Cash, cash equivalents, and restricted cash an...",1,2024,left_only
